# Cross-validation and search

**P1 Core · D3 Synthesis · 100 minutes**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
rng = np.random.default_rng(23)
groups = np.repeat(np.arange(30), 20)
group_target = rng.integers(0, 2, size=30)
y = group_target[groups]
frame = pd.DataFrame({'group': groups.astype(str), 'weak_signal': y + rng.normal(scale=2.5, size=len(y))})

## Task

Compare ordinary stratified folds with group-held-out folds. The group identity is deliberately learnable in random-row folds and unseen in group folds.

In [ ]:
def compare_folds(frame, y, groups):
    preprocess = ColumnTransformer([
        ('group', OneHotEncoder(handle_unknown='ignore'), ['group']),
        ('signal', StandardScaler(), ['weak_signal'])])
    model = Pipeline([('preprocess', preprocess), ('model', LogisticRegression(max_iter=1000))])
    random_cv = StratifiedKFold(5, shuffle=True, random_state=23)
    group_cv = GroupKFold(5)
    random_score = cross_val_score(model, frame, y, cv=random_cv, scoring='accuracy')
    group_score = cross_val_score(model, frame, y, groups=groups, cv=group_cv, scoring='accuracy')
    return random_score, group_score

In [ ]:
random_score, group_score = compare_folds(frame, y, groups)
assert random_score.mean() > group_score.mean() + 0.15
{'random_mean': random_score.mean(), 'group_mean': group_score.mean(),
 'group_spread': group_score.std()}

## Transfer

Use chronological air-quality validation, place feature selection inside the pipeline, run a fixed-budget random search, and preserve one final time block.